# Auditoria de fontes para o piloto

## Resumo
Acesso e cruzamento demonstrados; recorte temporal e gênero ainda bloqueados por metadados ausentes. Nenhuma hipótese musical testada. Este caderno acompanha docs/relatorio-piloto-fontes.md.


## Contexto e método
Grão: linhas de cada snapshot, com contagens de IDs e vídeos distintos reportadas separadamente. Cobertura por faixas de 1, 10 e 100 milhões de visualizações históricas; faixas de auditoria, não grupos finais. Fontes, hashes e datas declaradas: data/source-manifest.json. Dados brutos não versionados. Execute o script de download primeiro.


In [1]:
from pathlib import Path
import importlib.util, json
root = Path.cwd()
if not (root / 'scripts/audit_sources.py').exists():
    root = root.parent
spec = importlib.util.spec_from_file_location('audit_sources', root / 'scripts/audit_sources.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
result = module.audit(root / 'data/raw/spotify_tracks.csv', root / 'data/raw/spotify_youtube.csv')
for name in ['spotify_tracks', 'spotify_youtube']:
    print(name, 'linhas:', result[name]['rows'], 'IDs distintos:', result[name]['distinct_nonempty_keys'])


spotify_tracks linhas: 114000 IDs distintos: 89741
spotify_youtube linhas: 20718 IDs distintos: 18862


## Resultados de cobertura
Os números abaixo contam linhas, incluindo repetições. Não representam a amostra final. A tabela pequena permite conferir numeradores e denominadores sem um gráfico adicional.


In [2]:
for band, values in result['coverage_by_audience_band'].items():
    print(band, f"{values['matched_rows']}/{values['candidate_rows']} = {values['exact_id_coverage_pct']:.2f}%")
print('IDs distintos correspondentes:', result['join']['matched_distinct_track_ids'])
print('Linhas de uma junção ingênua:', result['join']['naive_inner_join_rows'])


01_under_1m 524/4080 = 12.84%
02_1m_to_under_10m 782/4950 = 15.80%
03_10m_to_under_100m 1669/7293 = 22.89%
04_100m_or_more 1400/3925 = 35.67%
IDs distintos correspondentes: 3938
Linhas de uma junção ingênua: 7166


## Encaminhamento
O cruzamento tem cobertura desigual por audiência. Investigar a fonte pareada com enriquecimento independente de gênero e datas; conferir gravações antes de incluí-las. A ausência desses campos impede aprovar o desenho completo nesta etapa.

Validação: células executadas sequencialmente em Python; não executadas em kernel Jupyter e sem inspeção visual em notebook viewer neste ambiente. Para validação local completa, abrir e executar todas as células em Jupyter a partir da raiz ou da pasta notebooks.
